## UNIVARIATE IMPUTER

### Important Library Imports

In [ ]:
import pandas
import numpy
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
import pandas
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

pandas.set_option('display.max_columns', None)

In [ ]:
df = pandas.read_csv('https://raw.githubusercontent.com/campusx-official/100-days-of-machine-learning/refs/heads/main/day35-complete-case-analysis/data_science_job.csv')
df.sample(5)

### These techniques work seperatly for both categorical and numerical data.

### CURRENTLY USING PANDAS FOR THIS PURPOSE

In [ ]:
df['gender'] = df['gender'].fillna('Missing') #23%
df['enrolled_university'] = df['enrolled_university'].fillna(df['enrolled_university'].mode()[0]) #2%
df['education_level'] = df['education_level'].fillna(df['education_level'].mode()[0]) #2.4%
df['major_discipline'] = df['major_discipline'].fillna('Missing') #14.6%
df['company_type'] = df['company_type'].fillna('Missing') #32.6%
df['training_hours'] = df['training_hours'].fillna(df['training_hours'].mean()) #3.9%
df['experience'] = df['experience'].fillna(df['experience'].mean()) #0.3%
df['city_development_index'] = df['city_development_index'].fillna(df['city_development_index'].mean()) #2.5%
df['company_size'] = df['company_size'].fillna('Missing') #30%


### Build a sklearn Transformer rather than Pandas for transformation

In [ ]:
print('''
                       RAW DATAFRAME (df)
               [ Contains Missing Values (NaN) ]
                              │
                              ▼
                 ┌───────────────────────────┐
                 │     ColumnTransformer     │
                 └─────────────┬─────────────┘
                               │
         ┌─────────────────────┼─────────────────────┐
         ▼                     ▼                     ▼
   [ mean_cols ]        [ missing_cols ]       [ mode_cols ]
  • training_hours     • gender               • enrolled_university
  • experience         • major_discipline     • education_level
  • city_dev_index     • company_type
                       • company_size
         │                     │                     │
         ▼                     ▼                     ▼
┌─────────────────┐   ┌─────────────────┐   ┌─────────────────┐
│  SimpleImputer  │   │  SimpleImputer  │   │  SimpleImputer  │
│ strategy='mean' │   │fill_value='Miss'│   │ 'most_frequent' │
└────────┬────────┘   └────────┬────────┘   └────────┬────────┘
         │                     │                     │
         └─────────────────────┼─────────────────────┘
                               │
                               ▼
                    [ remainder='passthrough' ]
                     (Keeps untouched columns)
                               │
                               ▼
                      NUMPY ARRAY OUTPUT
             [ Cleaned Data (No Missing Values) ]
''')

In [ ]:
#Build a sklearn Pipeline by implying all the above pandas transformations
mean_cols = ['training_hours', 'experience', 'city_development_index']
missing_cols = ['gender', 'major_discipline', 'company_type', 'company_size']
mode_cols = ['enrolled_university' ,'education_level']

preprocessor = ColumnTransformer(
    transformers=[
        ('mean_impute', SimpleImputer(strategy='mean'), mean_cols),
        ('missing_impute', SimpleImputer(strategy='constant', fill_value='Missing'), missing_cols),
        ('mode_impute', SimpleImputer(strategy='most_frequent'), mode_cols)
    ],
    remainder='passthrough' # Keep other columns that are not transformed
)

pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

transformed_data = pipeline.fit_transform(df)

# Convert Numpy to Pandas
data = pandas.DataFrame(transformed_data, columns=df.columns)
data.sample(5)

In [ ]:
df.isnull().mean() * 100

## These techniques works well both types of Data..

### Using Missing Indicator in skLearn
##### ***MissingIndicator Class*** - Create a new column which has True/False values that explictly tell that if the value is filled or not. It doesn't changes the actual column and its missing value and we can't even pass this column to the model.

##### ***Using Implace [add_indicator=True]***  - then this fill the existing column with approaches used above and then create a new column with 0 & 1.

In [ ]:
from sklearn.impute import MissingIndicator

missing_indicator = MissingIndicator(features='all' , error_on_new=True)#Applies to all the columns of the Dataset
missing_indicator.fit_transform(df)
